# 附录：从研究到实战 —— 补充模块

> 本文档是《30天因子研究计划》的补充内容，覆盖原课程中缺失的实盘关键模块。
> 建议在完成第28天（ML选股）后阅读，在第30天（终极项目）中实践。

---

## A. 回测框架设计要点

### A.1 为什么不能用简单累加做回测？

课程中的向量化回测（每天对所有股票算收益然后累加）有以下问题：

1. **无法处理调仓日的不连续交易**：真实调仓日股价可能跳空
2. **无法处理涨跌停**：涨跌停板上的股票无法以收盘价成交
3. **无法处理停牌**：停牌股票不能买卖
4. **无交易成本**：印花税、佣金、滑点全部忽略
5. **无资金约束**：假设无限资金，可以买所有信号股票

### A.2 最小可行的回测框架


In [ ]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Dict, List

@dataclass
class BacktestConfig:
    initial_capital: float = 1_000_000       # 初始资金
    commission_rate: float = 0.0003           # 佣金率（双边）
    stamp_tax_rate: float = 0.0005            # 印花税（卖出单边）
    slippage_rate: float = 0.001              # 滑点率
    max_positions: int = 50                   # 最大持仓数
    max_position_pct: float = 0.05            # 单票最大仓位
    rebalance_freq: int = 20                  # 调仓频率（交易日）
    limit_up_down_filter: bool = True         # 是否过滤涨跌停

def simple_event_driven_backtest(
    prices: pd.DataFrame,
    signals: pd.DataFrame,
    config: BacktestConfig,
) -> pd.DataFrame:
    """
    简化的事件驱动回测引擎。
    
    参数：
    - prices: (T × N) 收盘价矩阵
    - signals: (T × N) 因子信号矩阵（越高越好）
    - config: 回测配置
    
    返回：每日组合净值、持仓、换手
    """
    dates = prices.index
    assets = prices.columns
    
    # 状态变量
    cash = config.initial_capital
    positions = pd.Series(0.0, index=assets)     # 持仓数量（股）
    nav_history = pd.Series(index=dates, dtype=float)
    turnover_history = pd.Series(index=dates, dtype=float)
    
    for i, date in enumerate(dates):
        if i == 0:
            nav_history[date] = cash
            continue
        
        prev_date = dates[i - 1]
        today_price = prices.loc[date].dropna()
        prev_price = prices.loc[prev_date].dropna()
        
        # 计算当前持仓市值
        common_assets = positions.index.intersection(today_price.index)
        portfolio_value = (positions[common_assets] * today_price[common_assets]).sum()
        total_value = cash + portfolio_value
        nav_history[date] = total_value
        
        # 判断是否调仓日
        if i % config.rebalance_freq != 0:
            continue
        
        # 生成目标权重（简化：Top N等权）
        today_signal = signals.loc[date].dropna()
        selected = today_signal.nlargest(config.max_positions).index
        target_weight = pd.Series(1.0 / len(selected), index=selected)
        
        # 计算换手
        current_weight = positions[common_assets] * today_price[common_assets] / total_value
        aligned_weight = target_weight.reindex(current_weight.index).fillna(0)
        turnover = (aligned_weight - current_weight).abs().sum() / 2
        turnover_history[date] = turnover
        
        # 模拟成交（简化：忽略涨跌停和流动性）
        target_positions = (target_weight * total_value / today_price[target_weight.index]).dropna()
        
        # 交易成本
        buy_value = target_positions[today_price[target_positions.index]] @ target_positions
        cost = buy_value * (config.commission_rate + config.slippage_rate)
        # 注意：这里简化了，真实应分买卖计算
        
        cash -= cost
        positions = target_positions  # 简化：直接调到目标仓位
    
    return pd.DataFrame({
        "nav": nav_history,
        "turnover": turnover_history,
    })

# 配置示例
# config = BacktestConfig(
#     initial_capital=1_000_000,
#     rebalance_freq=20,
#     max_positions=30,
# )
# result = simple_event_driven_backtest(close_panel, factor_signals, config)


### A.3 回测的常见陷阱

| 陷阱 | 表现 | 检查方法 |
|------|------|----------|
| 未来函数 | 回测收益异常高 | 逐日检查信号是否用到未来数据 |
| 幸存者偏差 | 只用现存股票回测 | 确认使用了历史成分股 |
| 前视偏差 | 财务数据提前使用 | 确认披露滞后处理 |
| 过拟合 | 样本内很好，样本外很差 | 留出样本外测试期 |
| 交易成本低估 | 高换手策略"看起来"很好 | 加入合理成本后重新评估 |

---

## B. 实盘考量清单

### B.1 信号生成到执行的延迟


T日收盘 → 计算因子 → 生成信号 → T+1日开盘买入


关键问题：
- 如果因子计算需要T日数据，最早T+1日才能交易
- 财报公告日 ≠ 数据可用日（Wind/Choice通常滞后1-3天）
- 停复牌、涨跌停导致信号无法执行

### B.2 仓位管理


In [ ]:
def position_sizing(signals: pd.Series, max_weight: float = 0.05, max_total: float = 1.0):
    """
    仓位管理：限制单票最大仓位和总仓位。
    """
    # 按信号强度排序
    ranked = signals.sort_values(ascending=False).dropna()
    
    weights = pd.Series(0.0, index=signals.index)
    remaining = max_total
    n_selected = 0
    
    for asset, signal in ranked.items():
        weight = min(max_weight, remaining)
        if weight < 0.005:  # 最小仓位阈值
            break
        weights[asset] = weight
        remaining -= weight
        n_selected += 1
    
    return weights, n_selected


### B.3 风控规则


In [ ]:
def risk_control_checks(positions, prices, limits):
    """
    风控检查清单。
    
    limits = {
        "max_drawdown": 0.20,       # 最大回撤20%
        "max_leverage": 1.0,        # 最大杠杆
        "max_industry_exposure": 0.30,  # 单一行业最大暴露30%
        "stop_loss_daily": 0.05,    # 单日止损5%
    }
    """
    warnings = []
    
    # 检查最大回撤
    current_dd = compute_drawdown(positions, prices)
    if current_dd < -limits["max_drawdown"]:
        warnings.append(f"⚠️ 回撤超限: {current_dd:.1%}")
    
    # 检查行业暴露
    industry_exposure = compute_industry_exposure(positions)
    max_ind = industry_exposure.max()
    if max_ind > limits["max_industry_exposure"]:
        warnings.append(f"⚠️ 行业暴露超限: {max_ind:.1%}")
    
    return warnings


### B.4 实盘上线前终检清单

- [ ] 回测使用**样本外数据**（最近1-2年未参与任何训练）
- [ ] 交易成本已包含：佣金 + 印花税 + 滑点 + 冲击成本
- [ ] 信号延迟已建模（T日信号 → T+1日执行）
- [ ] 涨跌停不可交易已处理
- [ ] 停牌股已从持仓中移除
- [ ] 仓位和行业集中度有上限约束
- [ ] 策略容量已估算，资金规模在容量范围内
- [ ] 有止损和回撤熔断机制
- [ ] 有每日绩效监控和异常报警
- [ ] 模拟盘运行至少1个月后再上实盘
- [ ] 实盘初期用小资金（< 总资金的10%）

---

## C. 学习路线图（总览）


In [ ]:
第0天：真实数据准备 ──────────────────────────────┐
                                                    │
第1-5天：方法论基础（框架→标签→IC→ICIR→分层）      │
第6-13天：经典因子（价值/质量/动量/波动率/流动性）  ├─ 用真实数据替换模拟数据
第14-15天：预处理（中性化/标准化）                  │
第16-20天：Alpha101导论与复现                      │
                                                    │
第21天：多因子合成（项目段入口）───────────────────┤
第22-25天：高级评估（衰减/拥挤/Barra）             │
第26-28天：ML因子（特征工程→模型训练→选股组合）    │
第29天：Alpha Zoo因子管理                          │
第30天：终极项目（面试级报告）                     │
                                                    │
附录：回测框架 + 实盘考量 ─────────────────────────┘
          ↓
    模拟盘验证（1个月）
          ↓
    小资金实盘
          ↓
    逐步放量


---

> 本附录内容仅用于量化研究学习，不构成投资建议。
